In [ ]:
"""
=====================================================================================
Проект: Автоматизация обработки данных ионной хроматографии (Ion Balance Automation)
Скрипт: Шаг 3 - Формирование финального сводного отчета (Pivot Report Generator)
=====================================================================================

Описание задачи:
Скрипт берет результаты математического расчета (файл Final_Balance.xlsx) и превращает 
их в матричный вид (Pivot Table), где строки - это пробы, а столбцы - ионы. Пустые 
ячейки заполняются значениями "ниже предела обнаружения" (< 0.10 и т.д.). 
Результат сохраняется в два листа Excel: в мг/л и мг-экв/л.

Входные данные:
- Самый свежий файл Final_Balance_*.xlsx (результат работы скрипта 02_balance_calc.py).

Выходные данные:
- Файл Summary_Ion_Balance_*.xlsx (сводная таблица для отчетов).
====================================================================================
"""

import pandas as pd        # Библиотека для работы с таблицами (DataFrame) и Excel-файлами
import datetime            # Модуль для получения текущей даты и времени (для названий файлов)
import os                  # Модуль для работы с файловой системой (проверка путей, времени создания)
import numpy as np         # Библиотека для математических операций и обработки пустых значений (NaN)
import glob                # Модуль для поиска файлов по маске (например, Final_Balance_*.xlsx)

# --- НАСТРОЙКИ ---
# Ищем ВСЕ файлы Final_Balance_*.xlsx в текущем каталоге и берем самый свежий
list_of_files = glob.glob('Final_Balance_*.xlsx')
if not list_of_files:
    print("❌ Ошибка: Не найдено ни одного файла 'Final_Balance_*.xlsx'. Сначала запустите скрипт 02_balance_calc.py!")
    exit()
# Сортируем файлы по дате изменения и берем последний (самый свежий)
input_file = max(list_of_files, key=os.path.getctime) 

# Пределы обнаружения (первая точка калибровки для каждого иона) согласно текущим документам ПНДФ и РД
limits = {
    'F': 0.01, 'Cl': 0.02, 'NO2': 0.05, 'Br': 0.08, 'NO3': 0.08,
    'PO4': 0.10, 'SO4': 0.10, 'Na': 0.10, 'NH4': 0.10, 'K': 0.10,
    'Mg': 0.10, 'Ca': 0.10, 'CO3': 6.10, 'HCO3': 6.00
}

# Полный список ионов для шапки таблицы (чтобы колонки шли в правильном порядке)
ion_order = ['F', 'Cl', 'NO2', 'Br', 'NO3', 'PO4', 'SO4', 'CO3', 'HCO3', 'Na', 'NH4', 'K', 'Mg', 'Ca']

if not os.path.exists(input_file):
    print(f"❌ Ошибка: Файл '{input_file}' не найден в папке.")
else:
    print(f"Загрузка данных из {input_file}...")
    df = pd.read_excel(input_file)

    # 1. Очистка данных
    # Удаляем строки с итоговыми суммами и пустыми строками, оставляем только реальные ионы
    df_clean = df[~df['Ион'].isin(['Σ Анионов', 'Σ Катионов', 'ОТКЛОНЕНИЕ, %']) & df['Sample'].notna()].copy()
    # Переименовываем расчетный натрий для красоты шапки
    df_clean['Ион'] = df_clean['Ион'].replace({'Na (по разности)': 'Na'})

    # 2. Создаем сводные таблицы (Pivot Tables)
    print("Формирование сводных таблиц...")
    # index='Sample' (строки - пробы), columns='Ион' (столбцы - ионы)
    pivot_mg = df_clean.pivot_table(index='Sample', columns='Ион', values='Конц. (мг/л)', aggfunc='first')
    pivot_meq = df_clean.pivot_table(index='Sample', columns='Ион', values='Конц. (мг-экв/л)', aggfunc='first')

    # 3. Принудительно создаем ВСЕ колонки в нужном порядке
    # reindex заставляет pandas создать колонки даже для тех ионов, которые вообще не встречались в данных (поставят NaN)
    pivot_mg = pivot_mg.reindex(columns=ion_order)
    pivot_meq = pivot_meq.reindex(columns=ion_order)

    # 4. Заменяем NaN (пустоты) на текст "< предел"
    print("Заполнение пустых ячеек пределами обнаружения...")
    for ion in ion_order:
        limit_val = limits.get(ion, '')
        # Если в ячейке NaN, пишем "< 0.10" (или другой предел). Если есть число - оставляем как есть.
        pivot_mg[ion] = pivot_mg[ion].apply(lambda x: f"< {limit_val}" if pd.isna(x) else x)
        pivot_meq[ion] = pivot_meq[ion].apply(lambda x: f"< {limit_val}" if pd.isna(x) else x)

    # 5. Сохранение с меткой времени
    now = datetime.datetime.now()
    timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")
    output_file = f'Summary_Ion_Balance_{timestamp}.xlsx'

    print(f"Сохранение результата в {output_file}...")
    
    # Записываем два листа в один Excel-файл
    with pd.ExcelWriter(output_file) as writer:
        pivot_mg.to_excel(writer, sheet_name='Сводная мг-л')
        pivot_meq.to_excel(writer, sheet_name='Сводная мг-экв-л')

    print("\n" + "="*50)
    print("✅ ГОТОВО!")
    print(f"Создан файл: {output_file}")
    print("="*50)

    # Показываем кусочек таблицы для проверки в Jupyter
    display(pivot_mg.head())